# Workspace Explorer

## Description
Explores the local workspace filesystem, analyzes project structure, searches for files or text, and summarizes the workspace layout. Use when you need to understand the project directory structure, locate specific files or configurations, find where variables/functions are defined, or get a general lay of the land in an unfamiliar repository.

## System Prompt
You are an autonomous workspace explorer sub-agent. Your goal is to rapidly and accurately comprehend the structure, contents, and purpose of the local project directory.

**Expected Input:**
The parent agent will provide a description of what to look for (e.g., 'summarize the project structure', 'find where the database connection is established', or 'list all markdown files').

**Capabilities & Best Practices:**
1. **Python Utilities First:** Use the `WorkspaceExplorer` Python class provided in the scratchpad cells below. It is designed to safely summarize large directories, respect common ignore patterns (like `node_modules` or `.git`), and avoid context-window overflows.
2. **Avoid Massive Outputs:** Never run unrestricted `find .` or `tree` commands, and avoid printing entire large files. Always use depth limits (`max_depth`) and read snippets.
3. **Fallback to Bash:** Use the `bash` tool (with `rg`, `fd`, `find`) if you need complex command-line searches not covered by the Python utilities.
4. **Read Files:** Use the `read_file` tool to inspect the exact contents of files once you locate them.
5. **Read-Only:** You are strictly read-only. Do not modify, move, or delete files.

**Final Output:**
Return a concise, structured markdown summary to the parent agent. Include the most relevant file paths, a high-level project architecture, and direct answers to the parent's specific queries.

In [8]:
import os
print('This is your current working directoy:', os.getcwd())

This is your current working directoy: /Users/nicolasfonteyne/.agents/subagents/tmp/workspace-explorer


In [10]:
# Reusable Workspace Explorer Python Utility
import re
from pathlib import Path
from collections import Counter
import json

class WorkspaceExplorer:
    def __init__(self, root="."):
        self.root = Path(root).resolve()
        # Common directories and files to ignore to keep context clean
        self.ignore_dirs = {'.git', 'node_modules', 'venv', '.venv', 'env', '__pycache__', '.pytest_cache', 'build', 'dist', '.next', 'coverage', '.idea', '.vscode'}
        self.ignore_exts = {'.pyc', '.pdf', '.png', '.jpg', '.jpeg', '.gif', '.zip', '.tar', '.gz', '.mp4', '.sqlite3', '.db', '.DS_Store'}

    def _is_ignored(self, path: Path) -> bool:
        if path.name in self.ignore_dirs:
            return True
        if path.suffix.lower() in self.ignore_exts:
            return True
        return False

    def get_tree(self, max_depth=2) -> str:
        """Generates a clean tree view of the directory up to max_depth."""
        tree_str = []
        
        def walk(current_path, prefix="", depth=0):
            if depth > max_depth:
                return
            try:
                entries = sorted(list(current_path.iterdir()), key=lambda x: (not x.is_dir(), x.name))
            except PermissionError:
                return
                
            entries = [e for e in entries if not self._is_ignored(e) and not e.name.startswith('.')]
            
            for i, entry in enumerate(entries):
                is_last = (i == len(entries) - 1)
                connector = "└── " if is_last else "├── "
                tree_str.append(f"{prefix}{connector}{entry.name}{'/' if entry.is_dir() else ''}")
                
                if entry.is_dir():
                    extension = "    " if is_last else "│   "
                    walk(entry, prefix + extension, depth + 1)
                    
        tree_str.append(f"📂 {self.root.name}/")
        walk(self.root)
        return "\n".join(tree_str)

    def get_stats(self) -> dict:
        """Returns a dictionary of file counts by extension and total size."""
        ext_counts = Counter()
        total_size = 0
        file_count = 0
        
        for dirpath, dirnames, filenames in os.walk(self.root):
            # Modify dirnames in-place to skip ignored directories
            dirnames[:] = [d for d in dirnames if d not in self.ignore_dirs and not d.startswith('.')]
            
            for f in filenames:
                p = Path(dirpath) / f
                if self._is_ignored(p) or f.startswith('.'):
                    continue
                ext = p.suffix.lower() or 'no_extension'
                ext_counts[ext] += 1
                file_count += 1
                try:
                    total_size += p.stat().st_size
                except OSError:
                    pass
                    
        return {
            "total_files": file_count,
            "total_size_mb": round(total_size / (1024 * 1024), 2),
            "top_extensions": dict(ext_counts.most_common(10))
        }
        
    def find_content(self, pattern: str, max_results=15) -> list:
        """Searches for a regex pattern in files, returning a list of matched lines."""
        matches = []
        try:
            regex = re.compile(pattern, re.IGNORECASE)
        except re.error:
            return ["Invalid regex pattern"]
            
        for dirpath, dirnames, filenames in os.walk(self.root):
            dirnames[:] = [d for d in dirnames if d not in self.ignore_dirs and not d.startswith('.')]
            
            for f in filenames:
                if len(matches) >= max_results:
                    return matches
                    
                p = Path(dirpath) / f
                if self._is_ignored(p) or f.startswith('.'):
                    continue
                    
                try:
                    with open(p, 'r', encoding='utf-8', errors='ignore') as file:
                        for line_no, line in enumerate(file, 1):
                            if regex.search(line):
                                matches.append(f"{p.relative_to(self.root)}:{line_no}: {line.strip()[:120]}")
                                if len(matches) >= max_results:
                                    break
                except Exception:
                    pass
        return matches

In [11]:
# Run this cell to instantly get a high-level overview of the project structure and stats
explorer = WorkspaceExplorer('../../../../Github_nicolasakf/quant-mono')

print("=== WORKSPACE TREE (Depth=1) ===")
print(explorer.get_tree(max_depth=1))

print("\n=== WORKSPACE STATS ===")
import json
print(json.dumps(explorer.get_stats(), indent=2))


=== WORKSPACE TREE (Depth=1) ===
📂 quant-mono/
├── Classes/
│   ├── Asset.py
│   ├── BBG.py
│   ├── CheckingProcedures.py
│   ├── FixedIncome.py
│   ├── K15Tools.py
│   ├── Log.py
│   ├── Metadata.py
│   ├── Pricing.py
│   ├── Reports.py
│   ├── Risk.py
│   ├── RiskMetrics.py
│   ├── SSHUtil.py
│   ├── Scenarios.py
│   ├── Signal.py
│   ├── Synthetics.py
│   ├── WLInputs.py
│   └── desktop.ini
├── DataServices/
│   ├── BCB/
│   ├── Binance/
│   ├── Bloomberg/
│   ├── Exegy/
│   ├── FRED/
│   ├── Goldman/
│   ├── Procedures/
│   ├── Reuters/
│   ├── Sqzme/
│   ├── UBS/
│   ├── benzinga/
│   ├── factset/
│   ├── refinitiv/
│   ├── web_scrappers/
│   ├── __init__.py
│   └── generic.py
├── K15Credentials/
│   ├── resources/
│   ├── __init__.py
│   ├── keys.py
│   └── setup.py
├── K15CythonExtensions/
│   ├── Teste/
│   └── setup.py
├── K15Lab/
│   ├── Backtester/
│   ├── Common/
│   ├── Dataset/
│   ├── Indicators/
│   ├── Model/
│   ├── OMS/
│   ├── OrderRouter/
│   ├── Signal/
│   ├── St